### Confirming GPU availability and recording both software and hardware informatio inn the  environment

In [21]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device Name (GPU):", torch.cuda.get_device_name(0))
    memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / 1e9
        if torch.cuda.is_available()
        else None
    )
    print("Memory (GPU):", memory_gb, "GB")

Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA Available: True
Device Name (GPU): Tesla T4
Memory (GPU): 15.637086208 GB


## Installing dependencies

In [22]:
%pip install -q -U transformers accelerate huggingface_hub pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 153.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but y

## Loading `Qwen/Qwen3-4B` model

In [23]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

model.eval()

print("Loaded model:", MODEL_ID)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-4B


## Let's inspect the downloaded model metadata

In [27]:
from pathlib import Path
from huggingface_hub import model_info

print("Model ID:", MODEL_ID)

# Inspect the model's cached configuration and snapshot revision.
cache_dir = Path("/root/.cache/huggingface/hub") / (
    "models--" + MODEL_ID.replace("/", "--")
)

snapshots_dir = cache_dir / "snapshots"

if snapshots_dir.exists():
    print("\nCached snapshot revisions:")
    for snapshot in snapshots_dir.iterdir():
        if snapshot.is_dir():
            print(snapshot.name)
else:
    print("No local snapshot directory found at:", snapshots_dir)

Model ID: Qwen/Qwen3-4B

Cached snapshot revisions:
1cfa9a7208912126459214e8b04321603b3df60c


## Checking the model repository

In [28]:
from huggingface_hub import model_info

info = model_info(MODEL_ID)

print("Model:", info.id)
print("Resolved revision:", info.sha)
print("License metadata:", info.cardData.get("license")
      if info.cardData else "Not available in card metadata")

Model: Qwen/Qwen3-4B
Resolved revision: 1cfa9a7208912126459214e8b04321603b3df60c
License metadata: apache-2.0


## Creating a reusable inference function

In [24]:
def generate_response(
    user_prompt,
    system_prompt,
    max_new_tokens=300
):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

## Verify the generation settings

In [29]:
import inspect

print(inspect.getsource(generate_response))

def generate_response(
    user_prompt,
    system_prompt,
    max_new_tokens=300
):
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()



## Testing the `Qwen/Qwen3-4B` model

In [25]:
test_prompt = """
A fictional hospital has limited computers and occasional
power interruptions. Recommend a practical administrative
workflow that respects these constraints.
"""

test_response = generate_response(
    test_prompt,
    "You assist with non-clinical hospital administration."
)

print(test_response)

In a fictional hospital with limited computers and occasional power interruptions, the administrative workflow must be designed to be **resilient, efficient, and adaptable**. Here's a practical administrative workflow that respects these constraints:

---

## **1. Paper-Based Documentation (Primary Method)**
- **Use of paper records** for all administrative tasks (patient charts, appointments, inventory, etc.).
- **Centralized filing system** in a secure, climate-controlled room to prevent damage from power outages or humidity.
- **Barcode labels** on files for easy retrieval and tracking.

---

## **2. Manual Data Entry and Updates**
- **Trained staff** to manually enter data into computers only when power is stable.
- **Backup paper copies** of all digital records to ensure continuity in case of system failure.
- **Scheduled data entry times** during periods of stable power (e.g., during the day when the hospital is less busy).

---

## **3. Emergency Power Solutions**
- **Backup gen

To test the model, I provide a test prompt and feed as input into the model, and from the response I got the `preliminary observations` is that the model performed well by recognizing the stated limitations, the model also did not assume resources that were not available, and also all the recommendations can be implementable and lastly, the model acknowledge uncertainty. This is a good sign to move on with the intended model. Note this is just an observation not my final findings.

## Building the evaluation dataset

Since this is a pilot-scale dataset, I will begin with 15 fictional scenarios datset(due to time constraints) which can demonstrate a research process, however it is too small to support broad claims about all language models or hospitals.

---

## The five constraint categories includes:

| Category | What the scenario tests |
|---|---|
| Power | Recommendations during electricity interruptions |
| Software failure | Responses when a digital system becomes unavailable |
| Computer availability | Workflows when staff share limited computers |
| Staff guidance | Responses when users lack sufficient software training |
| Incomplete digital records | Workflows involving paper records and incomplete electronic data |

I'll also include scenarios combining several constraints.

## Creating `dataset`

In [ ]:
import pandas as pd

scenarios = [
    {
        "id": "PWR01",
        "category": "power",
        "task": "Plan a workflow for routine administrative records.",
        "constraints": [
            "Electricity may be unavailable for several hours.",
            "Staff must preserve records and avoid duplicate entries."
        ]
    },
    {
        "id": "PWR02",
        "category": "power",
        "task": "Recommend a process for completing administrative work during outages.",
        "constraints": [
            "The facility has intermittent electricity.",
            "There is no budget for immediate infrastructure upgrades."
        ]
    },
    {
        "id": "PWR03",
        "category": "power",
        "task": "Suggest a continuity plan for a digital administrative workflow.",
        "constraints": [
            "Power interruptions are unpredictable.",
            "Staff must follow approved downtime procedures."
        ]
    },
    {
        "id": "SW01",
        "category": "software_failure",
        "task": "Recommend a response when hospital software becomes unavailable.",
        "constraints": [
            "The cause of the failure is unknown.",
            "The restoration time is unknown."
        ]
    },
    {
        "id": "SW02",
        "category": "software_failure",
        "task": "Plan how staff should respond to a failed software update.",
        "constraints": [
            "Staff cannot administer the software themselves.",
            "Authorized ICT support must approve technical changes."
        ]
    },
    {
        "id": "SW03",
        "category": "software_failure",
        "task": "Recommend a process for recovering interrupted administrative work.",
        "constraints": [
            "The application is temporarily unavailable.",
            "Existing records must not be overwritten or duplicated."
        ]
    },
    {
        "id": "CMP01",
        "category": "computer_availability",
        "task": "Design a workflow for staff entering routine digital records.",
        "constraints": [
            "Several staff members share limited computers.",
            "No new computers can be purchased this quarter."
        ]
    },
    {
        "id": "CMP02",
        "category": "computer_availability",
        "task": "Suggest a practical way to schedule digital administrative tasks.",
        "constraints": [
            "Only a few functioning workstations are available.",
            "Staff have competing administrative responsibilities."
        ]
    },
    {
        "id": "CMP03",
        "category": "computer_availability",
        "task": "Recommend a phased plan for expanding digital recordkeeping.",
        "constraints": [
            "Computer availability is limited.",
            "The hospital must continue its existing administrative work."
        ]
    },
    {
        "id": "STF01",
        "category": "staff_guidance",
        "task": "Recommend a plan for helping staff use a hospital application.",
        "constraints": [
            "Some staff have limited experience with the software.",
            "Only short training sessions can be scheduled."
        ]
    },
    {
        "id": "STF02",
        "category": "staff_guidance",
        "task": "Suggest a way to reduce repeated user errors in a digital workflow.",
        "constraints": [
            "Formal training resources are limited.",
            "Staff need simple instructions they can consult during work."
        ]
    },
    {
        "id": "STF03",
        "category": "staff_guidance",
        "task": "Plan the introduction of a new administrative software feature.",
        "constraints": [
            "Staff familiarity with the existing system varies.",
            "The workflow must remain operational during training."
        ]
    },
    {
        "id": "REC01",
        "category": "incomplete_records",
        "task": "Plan a transition from paper records to digital recordkeeping.",
        "constraints": [
            "Some historical records are incomplete.",
            "Existing records must not be silently discarded."
        ]
    },
    {
        "id": "REC02",
        "category": "incomplete_records",
        "task": "Recommend a process for reconciling paper and electronic records.",
        "constraints": [
            "Some records exist only on paper.",
            "Staff must identify potential duplicates."
        ]
    },
    {
        "id": "MIX01",
        "category": "combined",
        "task": "Propose a practical plan for improving routine administrative workflows.",
        "constraints": [
            "Electricity is unreliable.",
            "Computers are shared.",
            "Some staff need additional software guidance.",
            "Some records remain on paper."
        ]
    }
]

df = pd.DataFrame(scenarios)

print("Scenario count:", len(df))
display(df)

Scenario count: 15


,id,category,task,constraints
0,PWR01,power,Plan a workflow for routine administrative rec...,[Electricity may be unavailable for several ho...
1,PWR02,power,Recommend a process for completing administrat...,"[The facility has intermittent electricity., T..."
2,PWR03,power,Suggest a continuity plan for a digital admini...,"[Power interruptions are unpredictable., Staff..."
3,SW01,software_failure,Recommend a response when hospital software be...,"[The cause of the failure is unknown., The res..."
4,SW02,software_failure,Plan how staff should respond to a failed soft...,[Staff cannot administer the software themselv...
5,SW03,software_failure,Recommend a process for recovering interrupted...,"[The application is temporarily unavailable., ..."
6,CMP01,computer_availability,Design a workflow for staff entering routine d...,[Several staff members share limited computers...
7,CMP02,computer_availability,Suggest a practical way to schedule digital ad...,[Only a few functioning workstations are avail...
8,CMP03,computer_availability,Recommend a phased plan for expanding digital ...,"[Computer availability is limited., The hospit..."
9,STF01,staff_guidance,Recommend a plan for helping staff use a hospi...,[Some staff have limited experience with the s...


These scenarios are a starting point.They are inspired by the types of constraints I have expericed as an IT technician at the `Federal Medical Centre Jalingo, Taraba State, Nigeria`, not copied from FMC's internal systems.

## Saving `dataset`

In [ ]:
# import json

# dataset_path = "/content/evaluation_dataset.jsonl"

# with open(dataset_path, "w", encoding="utf-8") as f:
#     for scenario in scenarios:
#         f.write(json.dumps(scenario) + "\n")

# print("Saved:", dataset_path)

Saved: /content/evaluation_dataset.jsonl


## Loading the saved scenario `evaluation_dataset.jsonl` datasets

In [3]:
import os
import json
import pandas as pd

JSONL_PATH = "/content/evaluation_dataset.jsonl"
CSV_PATH = "/content/evaluation_dataset.csv"

if os.path.exists(JSONL_PATH):
    # Priority 1: Load the saved JSONL dataset.
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        scenarios = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    print(f"Loaded {len(scenarios)} scenarios from JSONL.")

elif os.path.exists(CSV_PATH):
    # Priority 2: Load the saved CSV dataset.
    scenarios = pd.read_csv(CSV_PATH).to_dict("records")

    # Convert constraints back to a list if stored as JSON strings.
    for scenario in scenarios:
        constraints = scenario.get("constraints", [])

        if isinstance(constraints, str):
            try:
                scenario["constraints"] = json.loads(constraints)
            except json.JSONDecodeError:
                scenario["constraints"] = [constraints] if constraints else []

    print(f"Loaded {len(scenarios)} scenarios from CSV.")

else:
    # Priority 3: Use the existing in-memory list.
    print("No saved dataset found. Using the existing scenarios list.")

print("Scenario count:", len(scenarios))
print("First scenario:")
print(scenarios[0] if scenarios else "No scenarios available.")

Loaded 15 scenarios from JSONL.
Scenario count: 15
First scenario:
{'id': 'PWR01', 'category': 'power', 'task': 'Plan a workflow for routine administrative records.', 'constraints': ['Electricity may be unavailable for several hours.', 'Staff must preserve records and avoid duplicate entries.']}


## Let's define the experiment to work on

I will compare two response methods.

1. **Condition A**: Standard prompting. The model receives the task and the explicit operational constraints.

It is asked to provide a practical plan, without a special constraint-checking procedure.

2. **Condition B**: Constraint-aware prompting. The model receives the same task and constraints but is instructed to:
    - Identify the stated constraints.
    - Identify missing information.
    - Draft a practical plan.
    - Check each proposed action against the constraints.
    - Revise conflicting recommendations.
    - State unresolved risks.

*The key difference is the additional response procedure. I am testing whether this procedure improves performance. It's not claiming that it changes the model's weights or permanently improves its capabilities.*

## Defining prompt

In [4]:
STANDARD_SYSTEM = """
You help with non-clinical hospital administrative workflows.

Recommend a practical plan for the task described by the user.
Explain immediate actions, longer-term improvements, dependencies,
and unresolved questions.

Do not make clinical decisions.
"""

CONSTRAINT_AWARE_SYSTEM = """
You help with non-clinical hospital administrative workflows.

Before recommending a plan:
1. List the explicit operational constraints.
2. Identify important missing information.
3. Draft a practical response.
4. Check the proposed actions against each constraint.
5. Revise any actions that conflict with the constraints.
6. Explain remaining risks and dependencies.

Present the final plan clearly.

Do not make clinical decisions or invent facts about the hospital.
"""

Notice that the standard instruction does not tell the model to perform the special constraint-checking procedure. This will help us isolate the intervention.

## Constructing the prompts

In [5]:
def make_prompt(row):
    constraints = "\n".join(
        f"- {c}" for c in row["constraints"]
    )

    return f"""
Task:
{row["task"]}

Operational constraints:
{constraints}

Recommend a practical plan for this fictional hospital scenario.
"""

We'll be will testing both response methods using the same prompt for each scenario. For 15 scenarios, this produces `30 responses` in total.

## Let's now run the experiment

In [6]:
import os
import time
import pandas as pd
from tqdm.auto import tqdm

OUTPUT_PATH = "/content/model_outputs.csv"

# Load the existing CSV if it is available; otherwise, start empty.
if os.path.exists(OUTPUT_PATH):
    results_df = pd.read_csv(OUTPUT_PATH)
    results = results_df.to_dict("records")
    print(f"Loaded {len(results)} existing responses from {OUTPUT_PATH}")
else:
    results = []
    results_df = pd.DataFrame()
    print("No existing CSV found. Starting a new experiment.")

# Identify scenario/method combinations already completed.
completed = {
    (str(row["scenario_id"]), str(row["method"]))
    for row in results
    if row.get("scenario_id") is not None
    and row.get("method") is not None
}

# Build the list of tasks that still need to run.
tasks = [
    (scenario, method, system_prompt)
    for scenario in scenarios
    for method, system_prompt in [
        ("standard", STANDARD_SYSTEM),
        ("constraint_aware", CONSTRAINT_AWARE_SYSTEM),
    ]
    if (str(scenario["id"]), method) not in completed
]

print(f"Total expected responses: {len(scenarios) * 2}")
print(f"Already completed: {len(completed)}")
print(f"Remaining to run: {len(tasks)}")

# Run only missing tasks and save after each response.
for scenario, method, system_prompt in tqdm(
    tasks,
    desc="Generating model responses",
    unit="response"
):
    prompt = make_prompt(scenario)
    start_time = time.time()

    try:
        response = generate_response(
            user_prompt=prompt,
            system_prompt=system_prompt,
            max_new_tokens=300
        )

        results.append({
            "scenario_id": scenario["id"],
            "category": scenario["category"],
            "method": method,
            "prompt": prompt,
            "response": response,
            "elapsed_seconds": round(time.time() - start_time, 2),
            "model_id": MODEL_ID
        })

        # Persist all existing and newly generated responses.
        pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)

    except Exception as e:
        print(
            f"\nError for scenario {scenario['id']} "
            f"({method}): {e}"
        )
        # Keep previous results safe and continue with the next task.
        continue

# Load the final saved CSV to verify the actual saved results.
results_df = pd.read_csv(OUTPUT_PATH) if os.path.exists(OUTPUT_PATH) else pd.DataFrame()

print(f"\nSaved responses: {len(results_df)}")
print(f"Output file: {OUTPUT_PATH}")
display(results_df.head())

# import time

# results = []

# for scenario in scenarios:
#     prompt = make_prompt(scenario)

#     for method, system_prompt in [
#         ("standard", STANDARD_SYSTEM),
#         ("constraint_aware", CONSTRAINT_AWARE_SYSTEM)
#     ]:
#         start_time = time.time()

#         response = generate_response(
#             user_prompt=prompt,
#             system_prompt=system_prompt,
#             max_new_tokens=300
#         )

#         results.append({
#             "scenario_id": scenario["id"],
#             "category": scenario["category"],
#             "method": method,
#             "prompt": prompt,
#             "response": response,
#             "elapsed_seconds": round(
#                 time.time() - start_time, 2
#             ),
#             "model_id": MODEL_ID
#         })

#         # Save progress after every response.
#         pd.DataFrame(results).to_csv(
#             "/content/model_outputs.csv",
#             index=False
#         )

# results_df = pd.DataFrame(results)

# print("Total responses:", len(results_df))
# display(results_df.head())

Loaded 30 existing responses from /content/model_outputs.csv
Total expected responses: 30
Already completed: 30
Remaining to run: 0


Generating model responses: 0response [00:00, ?response/s]


Saved responses: 30
Output file: /content/model_outputs.csv


,scenario_id,category,method,prompt,response,elapsed_seconds,model_id
0,PWR01,power,standard,\nTask:\nPlan a workflow for routine administr...,**Task: Plan a workflow for routine administra...,17.77,Qwen/Qwen3-4B
1,PWR01,power,constraint_aware,\nTask:\nPlan a workflow for routine administr...,**Task: Plan a workflow for routine administra...,17.46,Qwen/Qwen3-4B
2,PWR02,power,standard,\nTask:\nRecommend a process for completing ad...,**Task: Recommend a process for completing adm...,18.44,Qwen/Qwen3-4B
3,PWR02,power,constraint_aware,\nTask:\nRecommend a process for completing ad...,**Task: Recommend a process for completing adm...,18.55,Qwen/Qwen3-4B
4,PWR03,power,standard,\nTask:\nSuggest a continuity plan for a digit...,**Continuity Plan for Digital Administrative W...,18.06,Qwen/Qwen3-4B


## Inspecting model outputs

To make sure that the responses I got from the model is justified I want to look for specific evidence by answering these few questions:

- Did the standard response assume reliable electricity?
- Did the model recommend buying equipment when the scenario prohibited new purchases?
- Did it recommend troubleshooting actions beyond the staff's authority?
- Did it offer a sensible alternative when the software was unavailable?
- Did the constraint-aware procedure introduce unnecessary complexity or new mistakes?

In [7]:
for scenario_id in ["PWR01", "SW01", "CMP01"]:
    print("\n", "=" * 70)
    print("SCENARIO:", scenario_id)

    subset = results_df[
        results_df["scenario_id"] == scenario_id
    ]

    for _, row in subset.iterrows():
        print("\nMETHOD:", row["method"])
        print(row["response"])


SCENARIO: PWR01

METHOD: standard
**Task: Plan a workflow for routine administrative records in a hospital with operational constraints (electricity may be unavailable for several hours; staff must preserve records and avoid duplicate entries).**

---

### **Immediate Actions (0–24 hours)**

1. **Establish a Backup Power System:**
   - Ensure that critical systems (e.g., record-keeping, communication, and data entry) have access to backup generators or battery-powered devices.
   - Test the backup system to confirm it can sustain operations for at least 48 hours.

2. **Implement Paper-Based Record-Keeping for Critical Data:**
   - Designate a central location for paper-based record-keeping (e.g., a secure, climate-controlled room).
   - Use standardized forms for patient records, staff logs, and inventory tracking to ensure consistency and reduce duplication.

3. **Train Staff on Paper-Based Systems:**
   - Conduct training sessions to ensure all staff understand how to use paper-base

I'll have to record my observation as well since I don't want to label a recommendation a failure simply because it differs from my preferred solution.

## Scoring the results

This phase is critical because a model's response can sound reasonable without being feasible. So, I'll will use a four-part rubric as described in the table below.

| Criterion | Score 0 | Score 1 | Score 2 |
|---|---|---|---|
| Constraint adherence | Contradicts a stated constraint | Partially respects constraints | Respects all material constraints |
| Feasibility | Not realistically implementable | Partly feasible | Practical under the stated conditions |
| Assumption control | Makes major unsupported assumptions | Makes some unsupported assumptions | Avoids unjustified assumptions |
| Adaptation and specificity | Generic or unsuitable | Partially adapted | Clearly tailored to the scenario |

---

The maximum score is: `8 points` per response.

Also, I'll record whether the response contains a clear constraint violation.

  - 1 = at least one clear violation.
  - 0 = no clear violation identified.
  - NA = cannot reliably determine.

## Creating the scoring sheet

In [8]:
blank_scores = results_df[
    ["scenario_id", "category", "method", "response"]
].copy()

for column in [
    "constraint_adherence",
    "feasibility",
    "assumption_control",
    "adaptation_specificity",
]:
    blank_scores[column] = pd.NA

blank_scores["constraint_violation"] = pd.NA
blank_scores["reviewer_notes"] = ""

blank_scores.to_csv(
    "/content/manual_scores.csv",
    index=False
)

display(blank_scores.head())

,scenario_id,category,method,response,constraint_adherence,feasibility,assumption_control,adaptation_specificity,constraint_violation,reviewer_notes
0,PWR01,power,standard,**Task: Plan a workflow for routine administra...,<NA>,<NA>,<NA>,<NA>,<NA>,
1,PWR01,power,constraint_aware,**Task: Plan a workflow for routine administra...,<NA>,<NA>,<NA>,<NA>,<NA>,
2,PWR02,power,standard,**Task: Recommend a process for completing adm...,<NA>,<NA>,<NA>,<NA>,<NA>,
3,PWR02,power,constraint_aware,**Task: Recommend a process for completing adm...,<NA>,<NA>,<NA>,<NA>,<NA>,
4,PWR03,power,standard,**Continuity Plan for Digital Administrative W...,<NA>,<NA>,<NA>,<NA>,<NA>,


## Loading the `manual_scores_completed.csv`. I have manually scored all responses.

In [13]:
file_path = "/content/manual_scores_completed..csv"

if not os.path.exists(file_path) and os.path.exists("/content/manual_scores_completed..csv"):
    file_path = "/content/manual_scores_completed..csv"

scores = pd.read_csv(file_path)
scores.head(20)

,scenario_id,category,method,response,constraint_adherence,feasibility,assumption_control,adaptation_specificity,constraint_violation,reviewer_notes,total_score
0,PWR01,power,standard,**Task: Plan a workflow for routine administra...,1,1,1,2,1,Recommends backup generator/battery and paper ...,5
1,PWR01,power,constraint_aware,**Task: Plan a workflow for routine administra...,2,1,1,2,0,Identifies both constraints and proposes manua...,6
2,PWR02,power,standard,**Task: Recommend a process for completing adm...,0,1,1,1,1,Suggests securing a generator or battery despi...,3
3,PWR02,power,constraint_aware,**Task: Recommend a process for completing adm...,1,1,0,2,0,Recognizes the no-budget constraint but adds '...,4
4,PWR03,power,standard,**Continuity Plan for Digital Administrative W...,2,1,1,2,0,Focuses on continuity planning and impact asse...,6
5,PWR03,power,constraint_aware,**Task: Suggest a continuity plan for a digita...,2,1,2,2,0,Restates the constraints and asks relevant mis...,7
6,SW01,software_failure,standard,**Task:** Recommend a response when hospital s...,2,2,2,2,0,Recommends assessing outage scope rather than ...,8
7,SW01,software_failure,constraint_aware,**Step 1: List the explicit operational constr...,2,2,2,2,0,Preserves uncertainty and asks about system st...,8
8,SW02,software_failure,standard,**Task: Plan how staff should respond to a fai...,2,2,2,2,0,Recommends notifying staff and responding to f...,8
9,SW02,software_failure,constraint_aware,**Task: Plan how staff should respond to a fai...,2,2,2,2,0,Recognizes staff cannot administer software an...,8


## Calculating summary metrics

But before calculating our evaluation metrics, I want to verify that the scores loaded correctly.

In [14]:
score_columns = [
    "constraint_adherence",
    "feasibility",
    "assumption_control",
    "adaptation_specificity",
    "constraint_violation",
]

print("Rows:", len(scores))
print("Missing scores:")
print(scores[score_columns].isna().sum())

Rows: 30
Missing scores:
constraint_adherence      0
feasibility               0
assumption_control        0
adaptation_specificity    0
constraint_violation      0
dtype: int64


everything looks good. Let's start calculating the results.

In [15]:
scores["total_score"] = scores[
    score_columns
].sum(axis=1, min_count=len(score_columns))

summary = scores.groupby("method").agg(
    mean_total_score=("total_score", "mean"),
    mean_constraint_adherence=(
        "constraint_adherence", "mean"
    ),
    mean_feasibility=("feasibility", "mean"),
    mean_assumption_control=(
        "assumption_control", "mean"
    ),
    mean_adaptation_specificity=(
        "adaptation_specificity", "mean"
    ),
    responses=("scenario_id", "count"),
)

print(summary)

                  mean_total_score  mean_constraint_adherence  \
method                                                          
constraint_aware          6.866667                   1.866667   
standard                  6.533333                   1.666667   

                  mean_feasibility  mean_assumption_control  \
method                                                        
constraint_aware          1.533333                 1.466667   
standard                  1.333333                 1.333333   

                  mean_adaptation_specificity  responses  
method                                                    
constraint_aware                     2.000000         15  
standard                             1.933333         15  


Awesome, the calculations works, and the initial results suggest that the `constraint-aware` prompting method performed better than the `standard prompt` on the average scores I measured. Before interpreting this as my research finding, there are two important checks I still have to do: 1. to confirm how the scoring rubric was applied and 2. too calculate the constraint-violation rate separately. Lets proceed


## Results summary

| Metric                       | Standard | Constraint-aware | Difference |
| ---------------------------- | -------- | ---------------- | ---------- |
| Mean total score (maximum 8) | 6.53     | 6.87             | +0.33      |
| Constraint adherence (0-2)   | 1.67     | 1.87             | +0.20      |
| Feasibility (0-2)            | 1.33     | 1.53             | +0.20      |
| Assumption control (0-2)     | 1.33     | 1.47             | +0.13      |
| Adaptation specificity (0-2) | 1.93     | 2.00             | +0.07      |
| Responses scored             | 15       | 15               | —          |


The total score is the sum of four dimensions, each scored from 0 to 2, giving a maximum of 8. The constraint-aware method's mean is approximately 4.2% higher relative to the maximum possible score than the standard method's mean, based on the 0.33-point difference.

These are descriptive results from my 15 scenarios per method. They do not yet establish statistical significance or prove that constraint-aware prompting will improve hospital workflows in real-world settings.

## Let's calculate the `constraint-violation rates`

The current summary omits `constraint_violation`. And because the research question focuses on whether the model respects operational constraints, this is an important outcome to report.

In [16]:
# Calculate the proportion of responses with a constraint violation.
violation_summary = scores.groupby("method").agg(
    violations=("constraint_violation", lambda x: (x == 1).sum()),
    responses_with_violation_score=("constraint_violation", "count"),
)

violation_summary["violation_rate_pct"] = (
    100
    * violation_summary["violations"]
    / violation_summary["responses_with_violation_score"]
)

print(violation_summary)

                  violations  responses_with_violation_score  \
method                                                         
constraint_aware           0                              15   
standard                   4                              15   

                  violation_rate_pct  
method                                
constraint_aware            0.000000  
standard                   26.666667  


## Let's compare the methods scenario by scenario

Because both methods were tested on the same 15 scenarios, I want to also compare their paired scores rather than looking only at their separate averages.

In [17]:
# Pivot total scores so each scenario has both methods side by side.
paired = scores.pivot(
    index="scenario_id",
    columns="method",
    values="total_score"
).dropna(subset=["standard", "constraint_aware"])

paired["difference"] = (
    paired["constraint_aware"] - paired["standard"]
)

print("Scenarios compared:", len(paired))
print("Mean paired difference:", round(paired["difference"].mean(), 3))
print("Constraint-aware higher:", (paired["difference"] > 0).sum())
print("Standard higher:", (paired["difference"] < 0).sum())
print("Equal scores:", (paired["difference"] == 0).sum())

display(paired)

Scenarios compared: 15
Mean paired difference: 0.333
Constraint-aware higher: 4
Standard higher: 1
Equal scores: 10


method,constraint_aware,standard,difference
scenario_id,,,
CMP01,7,6,1
CMP02,6,6,0
CMP03,8,6,2
MIX01,6,6,0
PWR01,6,6,0
PWR02,4,4,0
PWR03,7,6,1
REC01,6,6,0
REC02,8,6,2


Because both methods were evaluated on the same scenarios, I want to also inspect exactly which scenarios produced the four standard-prompt violations. This will help me connect the quantitative result to concrete model behavior.

In [18]:
# Show constraint violations and compare both methods for each scenario.
violation_check = scores.pivot(
    index="scenario_id",
    columns="method",
    values="constraint_violation"
)

print("Scenarios with standard-prompt violations:")
display(violation_check[violation_check["standard"] == 1])

print("\nScenarios with constraint-aware violations:")
display(violation_check[violation_check["constraint_aware"] == 1])

Scenarios with standard-prompt violations:


method,constraint_aware,standard
scenario_id,,
CMP01,0,1
CMP02,0,1
PWR01,0,1
PWR02,0,1



Scenarios with constraint-aware violations:


method,constraint_aware,standard
scenario_id,,


The violations were concentrated in two operational categories: computer availability and power availability. This is worth investigating because both categories involve physical infrastructure constraints that a proposed digital workflow must respect.

However, the table alone does not establish why the standard prompt failed. So I need to examine the actual model responses and identify the specific recommendation that conflicted with each scenario's constraints.

## Let's extract the eight relevant responses

The cell belwo will retrieve both methods' responses for the four affected scenarios from the saved output file.

In [20]:
# import pandas as pd

outputs = pd.read_csv("/content/model_outputs.csv")

affected_scenarios = ["CMP01", "CMP02", "PWR01", "PWR02"]

comparison = outputs[
    outputs["scenario_id"].astype(str).isin(affected_scenarios)
][
    ["scenario_id", "method", "prompt", "response"]
].sort_values(["scenario_id", "method"])

for scenario_id in affected_scenarios:
    print(f"\n{'=' * 70}")
    print(f"SCENARIO: {scenario_id}")
    print("=" * 70)

    subset = comparison[comparison["scenario_id"] == scenario_id]

    for _, row in subset.iterrows():
        print(f"\nMETHOD: {row['method']}")
        print("-" * 50)
        print("PROMPT:")
        print(row["prompt"])
        print("\nRESPONSE:")
        print(row["response"])


SCENARIO: CMP01

METHOD: constraint_aware
--------------------------------------------------
PROMPT:

Task:
Design a workflow for staff entering routine digital records.

Operational constraints:
- Several staff members share limited computers.
- No new computers can be purchased this quarter.

Recommend a practical plan for this fictional hospital scenario.


RESPONSE:
**Task: Design a workflow for staff entering routine digital records**

---

### **1. Explicit Operational Constraints**

- **Limited computer resources**: Several staff members share a small number of computers.
- **No new computers can be purchased this quarter**: No additional hardware is available for expansion.
- **Shared workstations**: Staff must use the same computers for multiple tasks and users.
- **Routine digital records entry**: The task is repetitive and requires consistent access to the system.

---

### **2. Important Missing Information**

- **Number of staff members** and how they are distributed acro

## Overall Interpretation of `results`

### 1. Summary of findings

This pilot study evaluated whether constraint-aware prompting improves the ability of a small open-weight language model to recommend administrative workflows that respect operational constraints in hospital environments. Fifteen fictional scenarios were evaluated using two prompting methods: a standard prompt and a constraint-aware prompt. Each response was manually assessed across four dimensions: constraint adherence, feasibility, assumption control, and adaptation specificity.

The constraint-aware method achieved a mean total score of **6.87 out of 8**, compared with **6.53 out of 8** for the standard method, representing an average improvement of approximately **0.33 points**.

Across the individual scoring dimensions, the constraint-aware method achieved higher mean scores for constraint adherence (1.87 versus 1.67), feasibility (1.53 versus 1.33), assumption control (1.47 versus 1.33), and adaptation specificity (2.00 versus 1.93). Although these differences consistently favored the constraint-aware method, the magnitude of the average improvement in total score was modest.

### 2. Constraint violations

The clearest difference was observed in constraint violations. The standard method produced four responses classified as constraint violations out of 15, giving an observed violation rate of **26.7%**. The constraint-aware method produced no recorded violations among its 15 responses, giving an observed rate of **0%**.

The four violations occurred in scenarios CMP01, CMP02, PWR01, and PWR02, which concerned computer availability and power availability. This pattern suggests that explicitly directing a model to identify operational constraints and check its recommendations against them may help reduce recommendations that conflict with stated infrastructure limitations.

However, the absence of recorded violations in this small sample does not establish that the constraint-aware method will consistently avoid such errors in other scenarios. The actual model responses must be examined to understand the nature of the violations and verify that the scoring rubric was applied consistently.

### 3. Scenario-Level comparison

Comparing the two methods on the same scenarios showed that the constraint-aware method achieved a higher total score in four scenarios, a lower score in one scenario, and an equal score in ten scenarios.

The mean paired difference was approximately **+0.33 points** in favor of constraint-aware prompting. This indicates that the overall improvement was concentrated in a small number of scenarios rather than occurring uniformly across the dataset.

The results also show that the standard method performed equally well on many scenarios under the chosen rubric. Therefore, the findings do not suggest that constraint-aware prompting necessarily improves every aspect of every response. Instead, its potential value may be particularly relevant when tasks require the model to account explicitly for operational limitations.

### 4. Implications for `Hospital Administrative Workflows`

The findings are relevant to resource-constrained healthcare environments, where workflow recommendations may need to account for intermittent electricity, software interruptions, limited computer access, staff training needs, and incomplete digital records.

A recommendation that ignores these constraints may be impractical even if it appears reasonable in isolation. Prompting a model to identify constraints, examine dependencies, and review its proposed actions may provide a simple method for improving the practical relevance of its recommendations.

Nevertheless, this experiment evaluated fictional administrative scenarios rather than live hospital workflows. It did not measure improvements in patient outcomes, staff productivity, service delivery, or institutional costs. The results should therefore be interpreted as preliminary evidence about model response behavior, not as proof of operational effectiveness or clinical safety.

### 5. Limitations

Several limitations affect the interpretation of these findings:

- **Small sample size:** Only 15 scenarios were evaluated per method, limiting the generalizability of the results.
- **Synthetic scenarios:** The scenarios were fictional and may not capture the full complexity of actual hospital operations.
- **Manual scoring:** Scores depend on the interpretation and consistent application of the evaluation rubric.
- **Single-model evaluation:** The findings apply to the model and configuration tested; other models may behave differently.
- **Prompt sensitivity:** Results may depend on the exact wording of the system prompts, generation settings, and scenario descriptions.
- **No real-world validation:** The experiment did not test whether the recommendations could be implemented successfully in